# LeetCode 1969 — Minimum Non-Zero Product of the Array Elements bằng Python

Link đề: https://leetcode.com/problems/minimum-non-zero-product-of-the-array-elements/ · Độ khó: **Medium** · Chủ đề: Math, Greedy, Recursion

Skeleton LeetCode:

```python
class Solution:
    def minNonZeroProduct(self, p: int) -> int:
        ...
```

Notebook đi từ **vét cạn mọi cách phân bố bit** (với `p ≤ 3`) → nhận xét **mỗi cột bit giữ nguyên số bit 1** → **ghép cặp `x` với `2ᵖ−1−x`** rồi biến mỗi cặp thành `(1, 2ᵖ−2)` → tính tích bằng **số nguyên lớn** → bằng **lũy thừa nhanh theo modulo** (`pow(a, e, m)` và tự cài đặt).

## 1. Mục tiêu học

1. Hiểu phép "hoán đổi bit cùng vị trí giữa hai số" **bảo toàn số bit 1 ở mỗi cột** và **bảo toàn tổng** các số.
2. Thấy nhận xét tham lam: với tổng `a + b` cố định, tích `a·b` (khác 0) nhỏ nhất khi hai số **chênh nhau nhiều nhất** ⇒ đẩy về `1` và `a + b − 1`.
3. Ghép cặp `x ↔ 2ᵖ−1−x` để có `2ᵖ⁻¹ − 1` cặp, mỗi cặp thành `(1, 2ᵖ − 2)`; đáp án `(2ᵖ − 1) · (2ᵖ − 2)^(2ᵖ⁻¹ − 1) mod (10⁹+7)`.
4. Thành thạo **lũy thừa nhanh** (binary exponentiation) và lưu ý về **modulo của số mũ**.
5. Kiểm chứng công thức bằng vét cạn với `p` nhỏ.

## 2. Đề bài & ví dụ

Cho số nguyên dương `p`. Xét mảng `nums` gồm các số nguyên từ `1` tới `2ᵖ − 1` (viết ở dạng nhị phân `p` bit). Được thực hiện bao nhiêu lần tùy ý thao tác:
- chọn hai phần tử `x`, `y`;
- chọn **một vị trí bit**, hoán đổi bit đó của `x` với bit **cùng vị trí** của `y`.

Ví dụ `x = 1101`, `y = 0011`, đổi bit thứ 2 từ phải: `x = 1111`, `y = 0001`.

Tìm **tích khác 0 nhỏ nhất** của `nums` sau các thao tác, trả về **modulo `10⁹ + 7`** (tích nhỏ nhất được xác định **trước** khi lấy modulo).

**Ví dụ 1:** `p = 1` → `1` (`nums = [1]`).

**Ví dụ 2:** `p = 2` → `6` (`nums = [01, 10, 11]`; mọi hoán đổi hoặc làm tích bằng 0 hoặc không đổi).

**Ví dụ 3:** `p = 3` → `1512` (`[001, 110, 001, 110, 001, 110, 111]`, tích `1·6·1·6·1·6·7`).

**Ràng buộc:** `1 ≤ p ≤ 60`.

In [1]:
import random
import time
from itertools import combinations, product
from math import prod
from typing import List

MOD = 10**9 + 7
examples = [(1, 1), (2, 6), (3, 1512)]

def bits(x, p):
    return format(x, f'0{p}b')

p = 3
print([bits(x, p) for x in range(1, 2 ** p)])
print("cấu hình của ví dụ 3:", [bits(x, 3) for x in [1, 6, 1, 6, 1, 6, 7]], "tích =", prod([1, 6, 1, 6, 1, 6, 7]))

['001', '010', '011', '100', '101', '110', '111']
cấu hình của ví dụ 3: ['001', '110', '001', '110', '001', '110', '111'] tích = 1512


## 3. Kiến thức nền

### 3.1 Hoán đổi bit giữ nguyên số bit 1 ở mỗi cột

Viết các số thành bảng: hàng = số, cột = vị trí bit. Hoán đổi bit cùng vị trí giữa hai số = hoán đổi hai ô **trong cùng một cột** ⇒ số bit 1 của **mỗi cột** không đổi ⇒ **tổng** các số không đổi.

In [2]:
def col_counts(arr, p):
    return [sum((x >> b) & 1 for x in arr) for b in range(p - 1, -1, -1)]

orig = list(range(1, 8))
after = [1, 6, 1, 6, 1, 6, 7]
print("số bit 1 mỗi cột (gốc):", col_counts(orig, 3), "| sau:", col_counts(after, 3))
print("tổng:", sum(orig), sum(after))

số bit 1 mỗi cột (gốc): [4, 4, 4] | sau: [4, 4, 4]
tổng: 28 28


### 3.2 Cùng tổng, tích nhỏ nhất khi chênh lệch lớn nhất

Với `a + b = S` cố định và `a, b ≥ 1`: `a·b = a(S − a)` là parabol úp, nhỏ nhất ở biên `a = 1` ⇒ `a·b = S − 1`.

In [3]:
S = 7
print([(a, S - a, a * (S - a)) for a in range(1, S)])

[(1, 6, 6), (2, 5, 10), (3, 4, 12), (4, 3, 12), (5, 2, 10), (6, 1, 6)]


### 3.3 Lũy thừa nhanh theo modulo

`pow(a, e, m)` tính `aᵉ mod m` trong `O(log e)` phép nhân bằng cách bình phương liên tiếp. Tự cài đặt:

In [4]:
def power_mod(a, e, m):
    result, a = 1, a % m
    while e > 0:
        if e & 1:
            result = result * a % m
        a = a * a % m
        e >>= 1
    return result

print(power_mod(3, 200, MOD), pow(3, 200, MOD))
print("số mũ cực lớn 2^59 - 1:", power_mod(5, 2**59 - 1, MOD) == pow(5, 2**59 - 1, MOD))

136318165 136318165
số mũ cực lớn 2^59 - 1: True


### 3.4 Số nguyên lớn trong Python

Python tính chính xác `aᵉ` với số nguyên lớn, nhưng `(2ᵖ − 2)^(2ᵖ⁻¹ − 1)` có cỡ `p · 2ᵖ⁻¹` bit — với `p = 60` là `~3·10¹⁹` bit: **không thể** lưu. Chỉ dùng số lớn cho `p` nhỏ để kiểm chứng.

In [5]:
for p in range(1, 8):
    v = (2**p - 1) * (2**p - 2) ** (2 ** (p - 1) - 1)
    print(f"p={p}: tích chính xác có {v.bit_length()} bit")

p=1: tích chính xác có 1 bit
p=2: tích chính xác có 3 bit
p=3: tích chính xác có 11 bit
p=4: tích chính xác có 31 bit
p=5: tích chính xác có 79 bit
p=6: tích chính xác có 191 bit
p=7: tích chính xác có 447 bit


## 4. Ý tưởng nền tảng / trực giác

Ghép mỗi số `x` với **phần bù** của nó `x̄ = (2ᵖ − 1) − x` (đảo mọi bit). Với `x` từ `1` tới `2ᵖ − 2`, ta có `2ᵖ⁻¹ − 1` cặp `{x, x̄}`, mỗi cặp có tổng `2ᵖ − 1`; còn lại phần tử `2ᵖ − 1` (toàn bit 1) không có cặp.

Trong một cặp `{x, x̄}`, mỗi cột có **đúng một** bit 1. Hoán đổi các bit trong cặp, ta có thể dồn **mọi** bit 1 trừ bit thấp nhất sang một số: được `(1, 2ᵖ − 2)` — chênh lệch tối đa, tích nhỏ nhất `2ᵖ − 2` mà vẫn khác 0.

Tính tay `p = 3`: các cặp `{1, 6}`, `{2, 5}`, `{3, 4}`; mỗi cặp thành `(1, 6)`; thêm `7` ⇒ tích `1·6·1·6·1·6·7 = 6³·7 = 1512` ✔.

In [6]:
p = 3
M = 2**p - 1
pairs = [(x, M - x) for x in range(1, 2 ** (p - 1))]
print("các cặp {x, bù(x)}:", pairs, "| phần tử lẻ:", M)
print("sau khi biến đổi mỗi cặp thành (1, M-1):", [(1, M - 1)] * len(pairs))
print("tích =", M * (M - 1) ** len(pairs))

các cặp {x, bù(x)}: [(1, 6), (2, 5), (3, 4)] | phần tử lẻ: 7
sau khi biến đổi mỗi cặp thành (1, M-1): [(1, 6), (1, 6), (1, 6)]
tích = 1512


Minh họa nhận xét "đẩy chênh lệch làm tích giảm" trên cặp `(2, 5)` (tổng 7): nếu dồn **hết** bit sang số lớn ta được `(0, 7)` — tích bằng 0, bị cấm; phải giữ lại đúng **một** bit (bit thấp nhất) ở số nhỏ ⇒ `(1, 6)`, tích `6 < 10`.

In [7]:
a, b = 2, 5
print(f"  ({a}, {b}) = ({bits(a, 3)}, {bits(b, 3)}), tích {a * b}")
a, b = a ^ 0b010, b ^ 0b010
print(f"  đổi bit 1 -> ({a}, {b}) = ({bits(a, 3)}, {bits(b, 3)}), tích {a * b}  <- số 0: cấm, phải giữ một bit ở số nhỏ")
a, b = a ^ 0b001, b ^ 0b001
print(f"  đổi bit 0 -> ({a}, {b}) = ({bits(a, 3)}, {bits(b, 3)}), tích {a * b}  <- tối ưu cho cặp có tổng 7")

  (2, 5) = (010, 101), tích 10
  đổi bit 1 -> (0, 7) = (000, 111), tích 0  <- số 0: cấm, phải giữ một bit ở số nhỏ
  đổi bit 0 -> (1, 6) = (001, 110), tích 6  <- tối ưu cho cặp có tổng 7


## 5. Cách 1 — Brute Force / baseline: vét cạn mọi phân bố bit

Vì thao tác cho phép hoán vị tùy ý các bit **trong mỗi cột**, mọi cấu hình **giữ đúng số bit 1 mỗi cột** đều đạt được. Vét cạn: với mỗi cột chọn tập hàng nhận bit 1 (`C(m, c)` cách), ghép các cột, tính tích, lấy min khác 0. Chỉ khả thi với `p ≤ 3`.

In [8]:
def min_product_brute(p):
    m = 2**p - 1                         # số phần tử
    counts = col_counts(list(range(1, m + 1)), p)   # số bit 1 mỗi cột (từ bit cao xuống)
    col_choices = [list(combinations(range(m), c)) for c in counts]
    best = None
    for choice in product(*col_choices):
        vals = [0] * m
        for col, rows in enumerate(choice):
            bit = 1 << (p - 1 - col)
            for r in rows:
                vals[r] |= bit
        if 0 in vals:
            continue
        v = prod(vals)
        if best is None or v < best:
            best = v
    return best % MOD

for p_, exp in examples:
    t0 = time.perf_counter()
    print(f"p={p_}: {min_product_brute(p_)} (kỳ vọng {exp})  {(time.perf_counter() - t0) * 1000:.0f} ms")

p=1: 1 (kỳ vọng 1)  0 ms
p=2: 6 (kỳ vọng 6)  0 ms
p=3: 1512 (kỳ vọng 1512)  38 ms


**Độ phức tạp:** `Π_cột C(m, c)` cấu hình — với `p = 3` là `35³ ≈ 43 000`; `p = 4` đã là `6435⁴ ≈ 1.7·10¹⁵`: bất khả thi.

**Nút thắt:** ta cần **cấu trúc** của lời giải tối ưu — chính là ghép cặp với phần bù.

## 6. Ý tưởng thuật toán / bước đột phá

**Nhận xét 1 (tích khác 0):** không được tạo số 0 — mỗi số phải giữ ít nhất một bit 1.

**Nhận xét 2 (tham lam cục bộ):** với hai số `a ≤ b` có tổng cố định, đổi một bit để `a` giảm và `b` tăng cùng lượng làm tích **giảm** (vì `(a − d)(b + d) = ab − d(b − a) − d² < ab`). Lặp lại tới khi không giảm được nữa ⇒ trong mỗi cặp bù, được `(1, 2ᵖ − 2)`.

**Kết quả:** `ans = (2ᵖ − 1) · (2ᵖ − 2)^(2ᵖ⁻¹ − 1)`, lấy mod `10⁹ + 7`.

**Quy tắc cài đặt:** số mũ `2ᵖ⁻¹ − 1` có thể tới `~5.8·10¹⁷` ⇒ **bắt buộc** lũy thừa nhanh theo modulo; **không** được lấy số mũ mod `10⁹+7`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Mô phỏng trực tiếp phép ghép cặp (số nguyên lớn, `p` nhỏ)

Dựng mảng, ghép `x` với bù, đổi từng bit (trừ bit thấp nhất) sang số lớn hơn, rồi nhân bằng số nguyên lớn. Dùng để **minh họa** quá trình.

In [9]:
def min_product_simulate(p):
    M = 2**p - 1
    vals = {}
    for x in range(1, 2 ** (p - 1)):
        a, b = x, M - x
        # dồn mọi bit 1 (trừ bit 0) về b: với mỗi bit k >= 1 mà a có, đổi sang b
        for k in range(1, p):
            if (a >> k) & 1 and not (b >> k) & 1:
                a ^= 1 << k; b ^= 1 << k
        # bảo đảm a giữ bit 0 (nếu bit 0 đang ở b thì đổi)
        if not a & 1:
            a ^= 1; b ^= 1
        vals[x] = (a, b)
    arr = [v for pair in vals.values() for v in pair] + [M]
    assert sorted(col_counts(arr, p)) == sorted(col_counts(list(range(1, M + 1)), p))
    return prod(arr) % MOD, arr

for p_, exp in examples:
    v, arr = min_product_simulate(p_)
    print(f"p={p_}: {v} (kỳ vọng {exp}), mảng sau biến đổi: {sorted(arr)}")

p=1: 1 (kỳ vọng 1), mảng sau biến đổi: [1]
p=2: 6 (kỳ vọng 6), mảng sau biến đổi: [1, 2, 3]
p=3: 1512 (kỳ vọng 1512), mảng sau biến đổi: [1, 1, 1, 6, 6, 6, 7]


**Độ phức tạp:** `O(2ᵖ · p)` thao tác + nhân số lớn — chỉ dùng cho `p ≤ 20`.

### 7.2 Cách 3 — Công thức với số nguyên lớn

In [10]:
def min_product_bigint(p):
    return (2**p - 1) * (2**p - 2) ** (2 ** (p - 1) - 1) % MOD

for p_, exp in examples:
    print(f"p={p_}: {min_product_bigint(p_)} (kỳ vọng {exp})")
t0 = time.perf_counter(); v = min_product_bigint(20); dt = time.perf_counter() - t0
print(f"p=20 bằng số lớn: {v} ({dt * 1000:.0f} ms) — p lớn hơn sẽ quá chậm/tốn bộ nhớ")

p=1: 1 (kỳ vọng 1)
p=2: 6 (kỳ vọng 6)
p=3: 1512 (kỳ vọng 1512)
p=20 bằng số lớn: 212293935 (848 ms) — p lớn hơn sẽ quá chậm/tốn bộ nhớ


### 7.3 Cách 4 — Lũy thừa nhanh theo modulo (lời giải chính)

In [11]:
def min_product_fast(p):
    M = (2**p - 1) % MOD
    base = (2**p - 2) % MOD
    e = 2 ** (p - 1) - 1
    return M * power_mod(base, e, MOD) % MOD

for p_, exp in examples:
    print(f"p={p_}: {min_product_fast(p_)} (kỳ vọng {exp})")
print("p=60:", min_product_fast(60))

p=1: 1 (kỳ vọng 1)
p=2: 6 (kỳ vọng 6)
p=3: 1512 (kỳ vọng 1512)
p=60: 813987236


**Độ phức tạp:** `O(log e) = O(p)` phép nhân modulo; bộ nhớ `O(1)`.

## 8. TRACE từng bước

### 8.1 Trace phép ghép cặp với `p = 3`

In [12]:
def trace_pairs(p):
    M = 2**p - 1
    for x in range(1, 2 ** (p - 1)):
        a, b = x, M - x
        steps = [f"{bits(a, p)}/{bits(b, p)}"]
        for k in range(1, p):
            if (a >> k) & 1 and not (b >> k) & 1:
                a ^= 1 << k; b ^= 1 << k
                steps.append(f"đổi bit {k}: {bits(a, p)}/{bits(b, p)}")
        if not a & 1:
            a ^= 1; b ^= 1
            steps.append(f"đổi bit 0: {bits(a, p)}/{bits(b, p)}")
        print(f"  cặp ({x}, {M - x}): " + " -> ".join(steps) + f"  => ({a}, {b}), tích {a * b}")

trace_pairs(3)

  cặp (1, 6): 001/110  => (1, 6), tích 6
  cặp (2, 5): 010/101 -> đổi bit 1: 000/111 -> đổi bit 0: 001/110  => (1, 6), tích 6
  cặp (3, 4): 011/100 -> đổi bit 1: 001/110  => (1, 6), tích 6


### 8.2 Trace lũy thừa nhanh `14⁷ mod (10⁹+7)` (trường hợp `p = 4`: cơ số `2⁴ − 2 = 14`, số mũ `2³ − 1 = 7 = 111₂`)

Mỗi bước xét bit thấp nhất của số mũ: bit 1 ⇒ nhân `result` với `a`; sau đó bình phương `a` và dịch số mũ sang phải.

In [13]:
def trace_power(a, e, m):
    result = 1; a %= m; step = 0
    print(f"  tính {a}^{e} mod {m}")
    while e > 0:
        bit = e & 1
        if bit:
            result = result * a % m
        print(f"  bước {step}: e={bin(e)}, bit thấp={bit}, result={result}, a={a}")
        a = a * a % m; e >>= 1; step += 1
    return result

p = 4
r = trace_power(2**p - 2, 2 ** (p - 1) - 1, MOD)
print("  => (2^p - 1) * kết quả =", (2**p - 1) * r % MOD, "| kiểm số lớn:", min_product_bigint(4))

  tính 14^7 mod 1000000007
  bước 0: e=0b111, bit thấp=1, result=14, a=14
  bước 1: e=0b11, bit thấp=1, result=2744, a=196
  bước 2: e=0b1, bit thấp=1, result=105413504, a=38416
  => (2^p - 1) * kết quả = 581202553 | kiểm số lớn: 581202553


## 9. Code nộp LeetCode

In [14]:
class Solution:
    def minNonZeroProduct(self, p: int) -> int:
        MOD = 10**9 + 7
        largest = 2**p - 1                       # phần tử không có cặp
        pair_value = 2**p - 2                    # mỗi cặp {x, bù(x)} thành (1, 2^p - 2)
        pairs = 2 ** (p - 1) - 1                 # số cặp
        return largest % MOD * pow(pair_value % MOD, pairs, MOD) % MOD

for p_, exp in examples:
    assert Solution().minNonZeroProduct(p_) == exp
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

**Tập cấu hình đạt được.** Mỗi thao tác hoán đổi hai ô trong cùng một cột của bảng bit ⇒ mọi cấu hình giữ đúng số bit 1 mỗi cột đều đạt được (hoán vị bất kỳ trong một cột là tích các hoán đổi). Cột bit `k` có đúng `2ᵖ⁻¹` bit 1 (một nửa của `2ᵖ` số `0..2ᵖ−1`, và số 0 không có bit 1).

**Chặn dưới.** Xét cấu hình tối ưu (tích khác 0, mọi số `≥ 1`). Nếu có hai số `a ≤ b` với `2 ≤ a` và một vị trí bit mà `a` có 1, `b` có 0 — trừ trường hợp làm `a` thành 0 — đổi bit đó làm tích giảm nghiêm ngặt (mục 6) ⇒ mâu thuẫn tối ưu. Từ đó có thể chỉ ra cấu hình tối ưu gồm `2ᵖ⁻¹ − 1` số bằng `1`, `2ᵖ⁻¹ − 1` số bằng `2ᵖ − 2` và một số `2ᵖ − 1` (đếm bit: cột 0 có `2ᵖ⁻¹` bit 1 phân cho `2ᵖ⁻¹ − 1` số `1` và số `2ᵖ − 1`; mỗi cột `k ≥ 1` có `2ᵖ⁻¹` bit 1 phân cho `2ᵖ⁻¹ − 1` số `2ᵖ − 2` và số `2ᵖ − 1`). Kiểm chứng bằng vét cạn với `p ≤ 3` ở mục 12.

**Tính toán.** `(2ᵖ − 2)^(2ᵖ⁻¹ − 1) mod m` bằng lũy thừa nhanh là chính xác (phép nhân tương thích với mod); số mũ **không** được rút gọn theo `m`.

Kiểm chứng cấu trúc lời giải tối ưu bằng vét cạn với `p = 3`: liệt kê mọi cấu hình đạt tích nhỏ nhất và xem đa tập giá trị của chúng.

In [15]:
def optimal_multisets(p):
    m = 2**p - 1
    counts = col_counts(list(range(1, m + 1)), p)
    best, found = None, set()
    for choice in product(*[list(combinations(range(m), c)) for c in counts]):
        vals = [0] * m
        for col, rows in enumerate(choice):
            for r in rows:
                vals[r] |= 1 << (p - 1 - col)
        if 0 in vals:
            continue
        v = prod(vals)
        if best is None or v < best:
            best, found = v, {tuple(sorted(vals))}
        elif v == best:
            found.add(tuple(sorted(vals)))
    return best, found

for p_ in (2, 3):
    best, found = optimal_multisets(p_)
    print(f"p={p_}: tích nhỏ nhất {best}, các đa tập đạt được: {found}")

p=2: tích nhỏ nhất 6, các đa tập đạt được: {(1, 2, 3)}
p=3: tích nhỏ nhất 1512, các đa tập đạt được: {(1, 1, 1, 6, 6, 6, 7)}


## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn phân bố bit | `Π C(m, c)` (hàm mũ kép) | `O(2ᵖ)` |
| 2. Mô phỏng ghép cặp | `O(2ᵖ · p)` + nhân số lớn | `O(2ᵖ)` |
| 3. Công thức số lớn | `O(p · 2ᵖ)` bit | `O(p · 2ᵖ)` bit |
| 4. Lũy thừa nhanh | `O(p)` | `O(1)` |

In [16]:
for name, fn in [("fast", min_product_fast), ("Solution", Solution().minNonZeroProduct)]:
    t0 = time.perf_counter(); vals = [fn(p_) for p_ in range(1, 61)]; dt = time.perf_counter() - t0
    print(f"{name:<9}: p=1..60 trong {dt * 1000:.2f} ms, p=60 -> {vals[-1]}")

fast     : p=1..60 trong 0.19 ms, p=60 -> 813987236
Solution : p=1..60 trong 0.10 ms, p=60 -> 813987236


## 12. Bộ test

In [17]:
for p_, exp in examples:
    assert min_product_brute(p_) == exp == min_product_fast(p_) == Solution().minNonZeroProduct(p_)
print("Ví dụ đề (gồm vét cạn với p <= 3): OK")

for p_ in range(1, 16):
    a = min_product_bigint(p_)
    assert a == min_product_fast(p_) == Solution().minNonZeroProduct(p_) == min_product_simulate(p_)[0]
print("p = 1..15: số lớn = lũy thừa nhanh = mô phỏng ghép cặp: OK")

rnd = random.Random(1969)
for _ in range(500):
    a, e, m = rnd.randint(0, 10**18), rnd.randint(0, 10**18), rnd.randint(1, 10**9 + 7)
    assert power_mod(a, e, m) == pow(a, e, m)
print("power_mod khớp pow trên 500 bộ ngẫu nhiên: OK")
print("p=60 ->", Solution().minNonZeroProduct(60))

Ví dụ đề (gồm vét cạn với p <= 3): OK


p = 1..15: số lớn = lũy thừa nhanh = mô phỏng ghép cặp: OK
power_mod khớp pow trên 500 bộ ngẫu nhiên: OK
p=60 -> 813987236


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Rút gọn số mũ theo `MOD`

In [18]:
def min_product_bug_exp_mod(p):
    e = (2 ** (p - 1) - 1) % MOD                      # SAI: số mũ không được mod theo MOD
    return (2**p - 1) * pow(2**p - 2, e, MOD) % MOD

bad = [p_ for p_ in range(1, 61) if min_product_bug_exp_mod(p_) != Solution().minNonZeroProduct(p_)]
print("các p bị sai:", bad[:10], "... (tổng", len(bad), "giá trị)")

các p bị sai: [31, 32, 33, 34, 35, 36, 37, 38, 39, 40] ... (tổng 30 giá trị)


**Vì sao sai:** `a^e ≢ a^(e mod m)` nói chung. (Theo định lý Fermat nhỏ, có thể rút gọn theo `m − 1` **khi** `a` không chia hết cho `m` — nhưng không cần làm vậy vì `pow` đã nhanh.) Với `p ≤ 30`, `e < MOD` nên chưa lộ lỗi; từ `p ≈ 31` trở đi sai. **Sửa:** giữ nguyên số mũ.

### Lỗi 2 — Quên nhân phần tử `2ᵖ − 1`

In [19]:
def min_product_bug_no_max(p):
    return pow(2**p - 2, 2 ** (p - 1) - 1, MOD)       # SAI: thiếu phần tử lớn nhất

print("p=3: sai =", min_product_bug_no_max(3), "| đúng =", Solution().minNonZeroProduct(3))

p=3: sai = 216 | đúng = 1512


**Vì sao sai:** `2ᵖ − 1` không có cặp (bù của nó là 0, không có trong mảng) và giữ nguyên. **Sửa:** nhân thêm `(2ᵖ − 1) mod m`.

### Lỗi 3 — Ghép sai thành `(0, 2ᵖ − 1)` (tích bằng 0) hoặc ghép thành `(1, 2ᵖ − 1)`

In [20]:
def min_product_bug_pair(p):
    return (2**p - 1) * pow(2**p - 1, 2 ** (p - 1) - 1, MOD) % MOD   # SAI: cặp có tổng 2^p - 1 nên phần lớn là 2^p - 2

print("p=3: sai =", min_product_bug_pair(3), "| đúng =", Solution().minNonZeroProduct(3))

p=3: sai = 2401 | đúng = 1512


**Vì sao sai:** tổng một cặp là `2ᵖ − 1`; nếu một số là `1` thì số kia là `2ᵖ − 2`. Đẩy về `0` thì tích bằng 0 — đề cấm. **Sửa:** `(1, 2ᵖ − 2)`.

### Lỗi 4 — Độ ưu tiên toán tử: `2**p - 2 % MOD`

In [21]:
p = 3
print("2**p - 2 % MOD =", 2**p - 2 % MOD, "| (2**p - 2) % MOD =", (2**p - 2) % MOD)
p = 40
print("p=40:", 2**p - 2 % MOD, "vs", (2**p - 2) % MOD, "<- vế trái không được rút gọn (vẫn đúng giá trị nhưng không phải phần dư)")

2**p - 2 % MOD = 6 | (2**p - 2) % MOD = 6
p=40: 1099511627774 vs 511620081 <- vế trái không được rút gọn (vẫn đúng giá trị nhưng không phải phần dư)


**Vì sao nguy hiểm:** `%` ưu tiên hơn `−` nên `2**p - 2 % MOD` = `2**p − 2`; trong Python vẫn ra đúng sau khi `pow` lấy mod, nhưng trong ngôn ngữ khác (tràn số) hoặc khi kết hợp phép tính khác sẽ sai. **Sửa:** luôn đặt ngoặc `(2**p − 2) % MOD`.

### Lỗi 5 — Tính số lớn chính xác rồi mới mod

`(2**p − 2) ** (2**(p−1) − 1)` với `p = 60` cần `~3·10¹⁹` bit — treo máy/hết bộ nhớ. Ta **không chạy** thử với `p` lớn; mục 7.2 đã đo `p = 20`. **Sửa:** `pow(a, e, MOD)`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thao tác "hoán đổi bit cùng vị trí" ⇒ **bất biến theo cột**: số bit 1 mỗi cột, tổng các số.
- "Tích nhỏ nhất với tổng cố định" ⇒ đẩy **chênh lệch** lớn nhất (về `1` và phần còn lại); "tích lớn nhất" ⇒ cân bằng.
- Dãy `1..2ᵖ−1` ⇒ nghĩ tới **ghép cặp với phần bù** `x ↔ 2ᵖ−1−x`.
- Số mũ khổng lồ ⇒ `pow(base, exp, MOD)`; không mod số mũ (trừ khi dùng Fermat có điều kiện).
- Kiểm công thức bằng vét cạn với `p = 1, 2, 3`.

## 15. Biến thể của bài

1. **Tích lớn nhất** sau các thao tác: cân bằng các số trong cặp — tổng cố định, tích lớn nhất khi các số gần nhau (bài khác hẳn).
2. **Tổng bình phương nhỏ nhất/lớn nhất**: tổng cố định ⇒ nhỏ nhất khi đều, lớn nhất khi lệch.
3. **Mảng bất kỳ** (không phải `1..2ᵖ−1`): đếm bit 1 mỗi cột; tham lam phân bit để có nhiều số nhỏ nhất (LeetCode 2897 — Apply Operations on Array to Maximize Sum of Squares, dạng tương tự với AND/OR).
4. **Modulo khác** hoặc yêu cầu trả tích chính xác (khi `p` nhỏ).

In [22]:
def max_sum_squares_after_ops(nums, k):
    """LeetCode 2897-like: phép AND/OR giữ số bit 1 mỗi cột; dồn bit để có các số lớn nhất, lấy k số lớn nhất."""
    cnt = [0] * 31
    for x in nums:
        for b in range(31):
            cnt[b] += (x >> b) & 1
    total = 0
    for _ in range(k):
        v = 0
        for b in range(31):
            if cnt[b]:
                cnt[b] -= 1; v |= 1 << b
        total += v * v
    return total % MOD

print("2897 ví dụ [2,6,5,8], k=2 ->", max_sum_squares_after_ops([2, 6, 5, 8], 2))
p_small = 3
arr = list(range(1, 2**p_small))
print("p=3, tổng bình phương lớn nhất (dồn bit vào ít số):", max_sum_squares_after_ops(arr, len(arr)))

2897 ví dụ [2,6,5,8], k=2 -> 261
p=3, tổng bình phương lớn nhất (dồn bit vào ít số): 196


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | mọi phân bố bit theo cột | hàm mũ kép | lớn | kiểm chứng `p ≤ 3` |
| 2. Mô phỏng ghép cặp | dồn bit trong cặp bù | `O(2ᵖ p)` | `O(2ᵖ)` | minh họa |
| 3. Công thức số lớn | tích chính xác rồi mod | `O(p 2ᵖ)` bit | lớn | `p ≤ 20` |
| 4. Lũy thừa nhanh | `pow(a, e, MOD)` | `O(p)` | `O(1)` | **nộp bài** |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `Solution().minNonZeroProduct(4)` bằng bao nhiêu? (Tính `15 · 14⁷`.)

**Bài 2 (tính tay):** Với `p = 3`, viết bảng bit của cấu hình tối ưu `[1, 6, 1, 6, 1, 6, 7]` và đếm số bit 1 mỗi cột, so với mảng gốc.

**Bài 3 (điền TODO):** Hoàn thiện `optimal_array(p)` trả về **một** cấu hình tối ưu (danh sách `2ᵖ − 1` số), kiểm bảo toàn số bit mỗi cột.

**Bài 4 (tự viết lại):** Viết lũy thừa nhanh **đệ quy** `power_rec(a, e, m)` (chia đôi số mũ) và kiểm với `pow`.

**Bài 5 (mini challenge):** Viết `min_product_any(nums)` cho mảng **bất kỳ** (số dương, độ dài nhỏ): tích khác 0 nhỏ nhất sau các thao tác hoán đổi bit cùng vị trí; kiểm bằng vét cạn phân bố bit với mảng ≤ 5 phần tử, ≤ 3 bit.

In [23]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def optimal_array(p):
    # TODO: [1] * (2^(p-1) - 1) + [2^p - 2] * (2^(p-1) - 1) + [2^p - 1]
    pass

# Bài 4 — TODO
def power_rec(a, e, m):
    # TODO: e == 0 -> 1 % m; h = power_rec(a, e // 2, m); h*h (nhân thêm a nếu e lẻ)
    pass

# Bài 5 — TODO
def min_product_any(nums):
    # TODO: vét cạn phân bố bit theo cột như min_product_brute nhưng với nums tùy ý
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1**
- Mức 1: `p = 4`: số lớn nhất 15, mỗi cặp thành `(1, 14)`.
- Mức 2: Có `2³ − 1 = 7` cặp.
- Mức 3: `15 · 14⁷ = 1 581 202 560` ⇒ mod `10⁹ + 7` được `581 202 553`.

**Bài 2**
- Mức 1: `1 = 001`, `6 = 110`, `7 = 111`.
- Mức 2: Cột cao: ba số 6 và số 7 ⇒ 4.
- Mức 3: Mỗi cột có 4 bit 1, giống mảng gốc.

**Bài 3**
- Mức 1: Ghép như mục 4.
- Mức 2: Dùng `col_counts` để kiểm.
- Mức 3: `prod(...) % MOD` phải bằng `Solution()`.

**Bài 4**
- Mức 1: `aᵉ = (a^(e/2))²` (nhân thêm `a` nếu lẻ).
- Mức 2: Độ sâu đệ quy `O(log e)` ≤ 60 — an toàn.
- Mức 3: So với `pow` trên số ngẫu nhiên.

**Bài 5**
- Mức 1: Số bit 1 mỗi cột là bất biến.
- Mức 2: Vét cạn `combinations` cho từng cột.
- Mức 3: Với `nums = 1..2ᵖ−1` phải khớp công thức.

## 19. Đáp án tham khảo cho bài tập

In [24]:
# Bài 1
print("Bài 1:", Solution().minNonZeroProduct(4), "| 15*14^7 % MOD =", 15 * 14**7 % MOD)
assert Solution().minNonZeroProduct(4) == 15 * 14**7 % MOD

# Bài 2
cfg = [1, 6, 1, 6, 1, 6, 7]
for x in cfg:
    print("  ", bits(x, 3))
print("  số bit 1 mỗi cột:", col_counts(cfg, 3), "| gốc:", col_counts(list(range(1, 8)), 3))
assert col_counts(cfg, 3) == col_counts(list(range(1, 8)), 3)

# Bài 3
def optimal_array_sol(p):
    k = 2 ** (p - 1) - 1
    return [1] * k + [2**p - 2] * k + [2**p - 1]

for p_ in range(1, 12):
    arr = optimal_array_sol(p_)
    assert len(arr) == 2**p_ - 1 and col_counts(arr, p_) == col_counts(list(range(1, 2**p_)), p_)
    assert prod(arr) % MOD == Solution().minNonZeroProduct(p_)
print("Bài 3 OK | p=3:", optimal_array_sol(3))

Bài 1: 581202553 | 15*14^7 % MOD = 581202553
   001
   110
   001
   110
   001
   110
   111
  số bit 1 mỗi cột: [4, 4, 4] | gốc: [4, 4, 4]
Bài 3 OK | p=3: [1, 1, 1, 6, 6, 6, 7]


In [25]:
# Bài 4
def power_rec_sol(a, e, m):
    if e == 0:
        return 1 % m
    h = power_rec_sol(a, e // 2, m)
    h = h * h % m
    return h * a % m if e & 1 else h

rnd = random.Random(4)
for _ in range(300):
    a, e, m = rnd.randint(0, 10**12), rnd.randint(0, 2**60), rnd.randint(1, 10**9 + 7)
    assert power_rec_sol(a, e, m) == pow(a, e, m)
print("Bài 4 OK")

# Bài 5
def min_product_any_sol(nums):
    m = len(nums)
    P = max(nums).bit_length()
    counts = col_counts(nums, P)
    best = None
    for choice in product(*[list(combinations(range(m), c)) for c in counts]):
        vals = [0] * m
        for col, rows in enumerate(choice):
            for r in rows:
                vals[r] |= 1 << (P - 1 - col)
        if 0 not in vals:
            v = prod(vals)
            best = v if best is None else min(best, v)
    return best

assert min_product_any_sol(list(range(1, 8))) == 1512
rnd = random.Random(5)
for _ in range(50):
    nums = [rnd.randint(1, 7) for _ in range(rnd.randint(1, 4))]
    r = min_product_any_sol(nums)
    assert r is not None and r <= prod(nums)
print("Bài 5 OK | [3, 5, 6] ->", min_product_any_sol([3, 5, 6]), "| [7, 7] ->", min_product_any_sol([7, 7]))

Bài 4 OK
Bài 5 OK | [3, 5, 6] -> 42 | [7, 7] -> 49


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template công thức + lũy thừa nhanh:**

```python
MOD = 10**9 + 7
largest = 2**p - 1
pairs = 2**(p - 1) - 1
return largest % MOD * pow((2**p - 2) % MOD, pairs, MOD) % MOD
```

**Template lũy thừa nhanh tự viết:**

```python
def power_mod(a, e, m):
    r, a = 1, a % m
    while e:
        if e & 1: r = r * a % m
        a = a * a % m; e >>= 1
    return r
```

**Checklist:**
- [ ] Mỗi cặp thành `(1, 2ᵖ − 2)` — không tạo số 0?
- [ ] Nhân thêm phần tử `2ᵖ − 1`?
- [ ] Số mũ `2ᵖ⁻¹ − 1` giữ nguyên (không mod theo `MOD`)?
- [ ] Dùng `pow(a, e, MOD)` (không tính số lớn)?
- [ ] Ngoặc đầy đủ `(2**p − 2) % MOD`?
- [ ] Đã kiểm `p = 1, 2, 3` bằng vét cạn?
- [ ] Chữ ký đúng: `def minNonZeroProduct(self, p: int) -> int`?